# Budgerigar：层级 Token Echo 双重评估

本 notebook 对新的层级 checkpoint 依次执行时间轴行为和句子内容保持评估。训练 loss 下降不构成通过依据。

In [ ]:
#@title 1. 更新项目与安装依赖
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib
if not Path(REPO_DIR).is_dir(): subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else: subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train,data]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [key for key in list(sys.modules) if key=='budgerigar' or key.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. 挂载 Drive 并定位层级 checkpoint
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
FEATURE_FINGERPRINT='f1f2ace085a17835' #@param {type:'string'}
TARGET_SPEAKER='arctic_slt' #@param {type:'string'}
FEATURE_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.features.{FEATURE_FINGERPRINT}.jsonl'
RUN_DIR=WORK_ROOT/'checkpoints'/f'hierarchical_echo_{TARGET_SPEAKER}_{FEATURE_FINGERPRINT}'
CHECKPOINT=RUN_DIR/'best.pt'
assert FEATURE_MANIFEST.is_file(),FEATURE_MANIFEST
assert CHECKPOINT.is_file(),CHECKPOINT
print(CHECKPOINT)

In [ ]:
#@title 3. 时间轴行为评估
MAX_PAIRS=32 #@param {type:'integer'}
import json
from budgerigar.evaluate_echo import evaluate_checkpoint
EVAL_DIR=RUN_DIR/'combined_evaluation'
behavior=evaluate_checkpoint(CHECKPOINT,FEATURE_MANIFEST,EVAL_DIR/'timeline',max_pairs=MAX_PAIRS)
print(json.dumps(behavior,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 内容保持评估
from budgerigar.evaluate_content import evaluate_content
content=evaluate_content(CHECKPOINT,FEATURE_MANIFEST,EVAL_DIR/'content',max_pairs=MAX_PAIRS,candidates=16)
print(json.dumps(content,ensure_ascii=False,indent=2))

In [ ]:
#@title 5. 联合阶段判定
combined_pass=bool(behavior['behavior_pass'] and content['content_pass'])
summary={'architecture':'hierarchical_token_echo','behavior_pass':behavior['behavior_pass'],'content_pass':content['content_pass'],'combined_pass':combined_pass,'retrieval_top1':content['retrieval_top1'],'correct_margin':content['correct_vs_shuffled_margin']}
print(json.dumps(summary,ensure_ascii=False,indent=2))
(EVAL_DIR/'combined_report.json').write_text(json.dumps({'summary':summary,'behavior':behavior,'content':content},ensure_ascii=False,indent=2),encoding='utf-8')
if not combined_pass: print('联合评估未通过：不要扩大训练，按失败指标继续修正结构或损失。')

In [ ]:
#@title 6. 保存元数据
from budgerigar.experiment import write_run_metadata
metadata=write_run_metadata(EVAL_DIR/'run_metadata.json',FEATURE_MANIFEST,summary,repository=REPO_DIR)
print(metadata.read_text(encoding='utf-8'))